In [1]:
import ollama 
import os
from tqdm import tqdm
import json
import signal
import argparse
import wandb
import pandas as pd

import sys
from collections import defaultdict
from sklearn.metrics import precision_score, recall_score, f1_score, accuracy_score,confusion_matrix
from pathlib import Path

import optuna
import re

/opt/conda/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
from PIL import Image
import matplotlib.pyplot as plt
import matplotlib.image as mpimg

In [3]:
sys.argv = [
    'notebook',  
    '--modelname', 'llama3.2-vision:90b',
    '--data', '/root/home/data',
    '--data_path_files','/mnt/Gbenga_Enemy/ramy/WACV-2025-Workshop-ViGIR',
    '--results_dir', '/mnt/Gbenga_Enemy/ramy/results',
    '--timeout', '20',
    '--model_unloading',
    '--valid_responses' ,'/mnt/Gbenga_Enemy/ramy/results/valid_llama3_90b.json',
    '--test_responses' ,'/mnt/Gbenga_Enemy/ramy/results/test_llama3_90b.json'
    
]

In [4]:
parser = argparse.ArgumentParser(description="A script to run V-LLMs on different image classification datasets")

In [5]:
parser.add_argument("--modelname", type=str, required=True, help="The name of the V-LLM model")
parser.add_argument("--data", type=str, required=True, help="Path to the data")
parser.add_argument("--data_path_files", type=str, required=True, help="Path to the image data dir")
parser.add_argument("--results_dir", type=str, required=True, help="Folder name to save results")
parser.add_argument("--timeout", type=int, default=40, help="time out duration to skip one sample")
parser.add_argument("--model_unloading", action="store_true", help="Enables unloading mode. Every 100 sampels it unloades the model from the GPU to avoid carshing.")
parser.add_argument("--valid_responses",type=str, required=True)
parser.add_argument("--test_responses",type=str, required=True)


args = parser.parse_args()

In [6]:
run = wandb.init(
    entity="ramytrm",
    project=f"WACV-2025-QuestionFiltering",
    name="run_test_" +args.modelname+"QuestionFiltering"
)

wandb: Using wandb-core as the SDK backend.  Please refer to https://wandb.me/wandb-core for more information.
wandb: Currently logged in as: ramytrm. Use `wandb login --relogin` to force relogin


In [7]:
def display_image(image_path):
    img = mpimg.imread(image_path)
    plt.figure(figsize=(20, 20))
    plt.imshow(img)
    plt.axis('off')
    plt.show()

In [8]:
best_dict_path=os.path.join(args.results_dir,'Best_Q_Best_Dict.json')
print(best_dict_path)

/mnt/Gbenga_Enemy/ramy/results/Best_Q_Best_Dict.json


In [9]:
model_name = args.modelname #'llama3.2-vision:90b'
ollama.pull(model_name)

timeout_duration = args.timeout

options= {  # new
            "seed": 123,
            "temperature": 0,
            "num_ctx": 2048, # must be set, otherwise slightly random output
        }

model_labels = {}
count = 0

In [10]:
file_path = os.path.join(args.valid_responses)
with open(file_path, 'r') as file:
    valid_responses = json.load(file)
    
    
file_path = os.path.join(args.test_responses)
with open(file_path, 'r') as file:
    test_responses = json.load(file)

In [11]:
data_reformated = {}

for key, item in valid_responses.items():
    q_and_a = []
    for i in range(len(item)):
        tmp = item[i][1:]   
        q_and_a.append(tmp)

    data_reformated[key] = q_and_a

In [12]:
data_reformated_test = {}

for key, item in test_responses.items():
    q_and_a = []
    for i in range(len(item)):
        tmp = item[i][1:]   
        q_and_a.append(tmp)

    data_reformated_test[key] = q_and_a

In [13]:
len(data_reformated)+len(data_reformated_test)

621

In [14]:
binary_vars=[1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1]

for index, (key, item) in enumerate(data_reformated.items()):

    formatted_qas = []
    for i, qa in enumerate(item):
        if binary_vars[i] == 0:
            continue
        formatted_qa = f"Q: {qa[0]}\nA: {qa[1]}"
        formatted_qas.append(formatted_qa)

    prompt = "\n".join(formatted_qas)

    question = "Based on the following questions and their answers, determine if there is evidence of an ephemeral gully in the observed area. Carefully analyze all the questions and the responses to assess.\n\n"
    question += prompt
    question += "\n\nAfter considering these responses, provide a clear conclusion: Is there evidence of an ephemeral gully? Answer with yes or no only."
    break

In [15]:
print(question)

Based on the following questions and their answers, determine if there is evidence of an ephemeral gully in the observed area. Carefully analyze all the questions and the responses to assess.

Q: Given these six images of the exact same area and collected over a period of 10 years, do you see a low point in the terrain? Answer with yes or no only!
A: No.
Q: Given these six images of the exact same area and collected over a period of 10 years, are there indications of nearby human activity, such as tillage or machinery tracks? Answer with yes or no only!
A: No.

After considering these responses, provide a clear conclusion: Is there evidence of an ephemeral gully? Answer with yes or no only.


In [16]:
def extract_first_yes_no(text):
    # Use regular expressions to find "Yes" or "No" (case-insensitive)
    match = re.search(r'\b(Yes|No)\b', text, re.IGNORECASE)
    if match:
        # Return the first match in its original case
        return match.group(0)
    return None

In [17]:
def evaluate_model(data_reform, responses, binary_vars, model_name, options,trial,VALID=0):
    ground_truth_test = []
    model_response_test = []

    for index, (key, item) in enumerate(data_reform.items()):

        formatted_qas = []
        for i, qa in enumerate(item):
            if binary_vars[i] == 0:
                continue
            formatted_qa = f"Q: {qa[0]}\nA: {qa[1]}"
            formatted_qas.append(formatted_qa)

        prompt = "\n".join(formatted_qas)

        question = "Based on the following questions and their answers, determine if there is evidence of an ephemeral gully in the observed area. Carefully analyze all the questions and the responses to assess.\n\n"
        question += prompt
        question += "\n\nAfter considering these responses, provide a clear conclusion: Is there evidence of an ephemeral gully? Answer with yes or no only."
#         question += "\n\nIs there evidence of an ephemeral gully? Answer with yes or no only."

        response = ollama.generate(model=model_name, prompt=question, options=options)
        text=extract_first_yes_no(response['response'])
        
        
        if text == 'Yes':
            model_response_test.append(1)
        else:
            model_response_test.append(0)
            
        if responses[key][0][0]['label'] == '4':
            ground_truth_test.append(1)
        else:
            ground_truth_test.append(0)
            
            
        if VALID==1 and index%50==0:
        
            macro_f1 = f1_score(ground_truth_test, model_response_test, average='macro')
            trial.report(macro_f1, step=index)

    return ground_truth_test, model_response_test

In [18]:
def log_best_dict(best_dict,file_path):

    wandb.log(best_dict)
    
    table = wandb.Table(data=[list(best_dict.values())], columns=list(best_dict.keys()))
    wandb.log({"best_dict_table": table})
    
    if os.path.isfile(file_path):
        artifact = wandb.Artifact("best_dict", type="dictionary")
        artifact.add_file(file_path)
        wandb.log_artifact(artifact)
    else:
        raise FileNotFoundError(f"The file {file_path} was not created successfully.")

In [19]:
def objective(trial):
    
    global best_dict, trail_no
    
    trail_no+=1
    
    print(trail_no,best_dict['best_macro'])
    
    binary_vars = [trial.suggest_int(f"x{i}", 0, 1) for i in range(15)]
    

    if sum(binary_vars)<=3:
        return 0
    
    ground_truth, model_response = evaluate_model(data_reformated, 
                                                  valid_responses, 
                                                  binary_vars, 
                                                  model_name, 
                                                  options,trial, 
                                                  VALID=1
                                                )
        
    
    macro_f1 = f1_score(ground_truth, model_response, average='macro')
    
    
    if macro_f1>best_dict['best_macro']:
        
        
        best_dict['all_best']=[]
        best_dict['all_best'].append(binary_vars)
        best_dict['best_macro']=macro_f1
        best_dict['best_questions']=binary_vars
        
        
        ground_truth_test, model_response_test = evaluate_model(
        data_reformated_test, test_responses, binary_vars, model_name, options, trial
        )


        macro_f1_test = f1_score(ground_truth_test, model_response_test, average='macro')
        best_dict['best_test']=macro_f1_test
        
        print('Got new best', best_dict['best_macro'],best_dict['best_test'])
        
     
    if macro_f1>=best_dict['best_macro']:
        with open(best_dict_path, 'w') as fp:
            json.dump(best_dict, fp, indent=4)
            
        log_best_dict(best_dict,best_dict_path)
        
    return macro_f1 

In [ ]:
trail_no=0
best_dict={}

best_dict['best_macro']=0
best_dict['best_questions']=[]
best_dict['all_best']=[]



study = optuna.create_study(direction="maximize")
study.optimize(objective, n_trials=1000)


print("Best objective value:", study.best_value)
print("Best parameters:", study.best_params)

[I 2024-11-15 01:59:43,962] A new study created in memory with name: no-name-23dfd6fe-40f9-439a-b774-c670b160183d


1 0


In [ ]:
tes=[1]*15

In [ ]:
print(len(tes))

In [ ]:
tes[1:12]=[0]*11

In [ ]:
print(tes)

In [ ]:
for index, (key, item) in enumerate(data_reformated.items()):
    formatted_qas = []
    for i, qa in enumerate(item):
        if tes[i]==0:
            continue
        formatted_qa = f"Q: {qa[0]}\nA: {qa[1]}"
        formatted_qas.append(formatted_qa)
        
    prompt = "\n".join(formatted_qas)
    print(prompt)
    print('________________________________')
    if index ==2:
        break

In [ ]:
data_reformated[str(356)]

In [ ]:
valid_responses

In [ ]:
binary_vars

In [ ]:
binary_vars=[1]*15

ground_truth, model_response = evaluate_model(data_reformated_test, 
                                              test_responses, 
                                              binary_vars, 
                                              model_name, 
                                              options, 12
                                            )


macro_f1 = f1_score(ground_truth, model_response, average='macro')

In [ ]:
macro_f1

In [ ]:
best_dict

In [ ]:
#     if macro_f1==best_dict['best_macro']:
#         best_dict['all_best'].append(binary_vars)
        
        
#         ground_truth_test, model_response_test = evaluate_model(
#             data_reformated_test, test_responses, binary_vars, model_name, options, trial
#         )


#         macro_f1_test = f1_score(ground_truth_test, model_response_test, average='macro')
#         best_dict['all_best_test'].append(macro_f1_test)
        
#         print('Got a new tie', best_dict['best_macro'],best_dict['best_test'])